In [ ]:
######## empieza analisis de predicciones

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

class Predictor:
    def __init__(self):
        self.scaler = StandardScaler()

    def load_and_preprocess_data(self, data_path):
        df = pd.read_csv(data_path)
        df['fecha'] = pd.to_datetime(df['fecha'])
        df = df.sort_values(by=['articulo', 'fecha'])
        return df

    def load_covariates_data(self, cov_path):
        return pd.read_csv(cov_path)

    def create_monthly_sales_data(self, df):
        df.loc[:, 'year'] = df['fecha'].dt.year
        df.loc[:, 'month'] = df['fecha'].dt.month

        monthly_data = df.groupby(['articulo', 'year', 'month']).agg({
            'cantidad': 'sum',
            'transacciones': 'sum',
            'venta_pen': 'sum',
            'fuente_suministro': 'first',
            'lt': 'first'
        }).reset_index()

        monthly_data['fecha'] = pd.to_datetime(monthly_data.apply(
            lambda row: pd.Timestamp(year=int(row['year']), month=int(row['month']), day=1), axis=1))
        monthly_data['lt'] = monthly_data['lt'].fillna(0)
        return monthly_data.sort_values(by=['articulo', 'fecha'])

    def select_features_with_lasso(self, X, y, feature_columns, alpha=0.01):
        """
        Perform feature selection using LASSO regression.
        Returns selected feature names based on non-zero coefficients.
        """
        X_scaled = self.scaler.fit_transform(X)
        
        lasso = Lasso(alpha=alpha, random_state=42)
        lasso.fit(X_scaled, y)
        
        selected_features = [feature for feature, coef in zip(feature_columns, lasso.coef_) 
                        if abs(coef) > 0]
                
        return selected_features

    def prepare_features_for_ml(self, all_monthly_data, df_cov, df_correlaciones_sig, sku):
        sku_correlations = df_correlaciones_sig[df_correlaciones_sig['sku'] == sku]
        
        if len(sku_correlations) == 0:
            return all_monthly_data[all_monthly_data['articulo'] == sku].copy(), []
            
        data = all_monthly_data[all_monthly_data['articulo'] == sku].merge(
            df_cov, on=['year', 'month'], how='left'
        )
        
        feature_columns = []
        for _, row in sku_correlations.iterrows():
            col_name = row['tipo']
            lag = row['lag']
            if lag > 0:
                data[f'{col_name}_lag_{lag}'] = data[col_name].shift(lag)
                feature_columns.append(f'{col_name}_lag_{lag}')
            else:
                feature_columns.append(col_name)
        
        data = data.dropna()
        
        if len(data) > 0 and len(feature_columns) > 0:
            X = data[feature_columns]
            y = data['cantidad']
            selected_features = self.select_features_with_lasso(X, y, feature_columns)
            return data, selected_features
        
        return data, []

    def weighted_mape(self, y_true, y_pred):
        errors = []
        for true, pred in zip(y_true, y_pred):
            if pd.isna(true) or pd.isna(pred) or true == 0:
                continue
            error = abs((true - pred) / true)
            if pred < true:  # underprediction
                error *= 2
            errors.append(error)
        return np.mean(errors) * 100 if errors else float('inf')

    def ES_forecast(self, series, alpha):
        series = np.array(series)
        result = [series[0]]
        for n in range(1, len(series)):
            result.append(alpha * series[n] + (1 - alpha) * result[n-1])
        return result[-1]

    def ES_opt_alpha(self, series):
        series = np.array(series)
        alpha_list = np.linspace(0.1, 0.9, 100)
        errors = []
        
        for alpha in alpha_list:
            error = []
            for i in range(1, len(series)-1):
                forecast = self.ES_forecast(series[:i], alpha)
                error.append(abs(forecast - series[i+1]))
            errors.append(np.mean(error))
        
        return alpha_list[np.argmin(errors)]

    def evaluate_models(self, data, feature_columns, target_col='cantidad', 
                    lookback_periods=[6, 12, None], val_year=None):
        best_score = float('inf')
        best_config = None
        best_model = None
        
        if len(data) < 4:
            return None, None, float('inf')
            
        # val_data = data[data['year'] == val_year] if val_year else data
        val_data = data
        val_size = 6
        
        for lookback in lookback_periods:
            train_data = (val_data.iloc[-(lookback+val_size):-val_size] 
                        if lookback else val_data.iloc[:-val_size])
            test_data = val_data.iloc[-val_size:]
            
            # if len(train_data) < 2:
            #     continue
                
            y_train = train_data['cantidad']
            y_test = test_data['cantidad']
            
            models = {
                'mean': y_train.mean(),
                'median': y_train.median(),
                'es': (self.ES_forecast, self.ES_opt_alpha(y_train))
            }
            
            if len(feature_columns) > 0:
                models.update({
                    'xgboost': xgb.XGBRegressor(random_state=42),
                    'linear': LinearRegression()
                })
                X_train = train_data[feature_columns]
                X_test = test_data[feature_columns]
            
            for model_name, model in models.items():
                try:
                    if model_name in ['xgboost', 'linear']:
                        model.fit(X_train, y_train)
                        y_pred = model.predict(X_test)
                    elif model_name == 'es':
                        forecast_func, alpha = model
                        y_pred = []
                        current_data = y_train.values
                        for _ in range(len(y_test)):
                            pred = forecast_func(current_data, alpha)
                            y_pred.append(pred)
                            current_data = np.append(current_data, pred)
                    else:  # mean or median
                        y_pred = [model] * len(y_test)
                    
                    y_pred = np.maximum(y_pred, 0)
                    score = self.weighted_mape(y_test, y_pred)
                    
                    if score < best_score:
                        best_score = score
                        best_config = (model_name, lookback)
                        best_model = model
                        
                except Exception as e:
                    continue
        
        return best_config, best_model, best_score

    def predict_future_months(self, data, feature_columns, best_model, 
                            best_model_name, lookback, num_months):
        predictions = []
        current_data = data.copy()
        current_data = current_data.sort_values(by="fecha")

        for _ in range(num_months):
            if best_model_name in ['xgboost', 'linear']:
                pred = best_model.predict(current_data[feature_columns].iloc[-1:])[0]
            elif best_model_name in ['mean', 'median']:
                last_6_months = current_data.iloc[-6:]  # Tomar solo los últimos 6 meses
                pred = last_6_months['cantidad'].mean() if best_model_name == 'mean' else last_6_months['cantidad'].median()
            else:  # ES
                forecast_func, alpha = best_model
                lookback_data = (current_data.iloc[-lookback:] if lookback 
                                else current_data)
                pred = forecast_func(lookback_data['cantidad'].dropna(), alpha)
            
            pred = np.maximum(pred, 0)
            predictions.append(pred)
            
            new_row = current_data.iloc[-1:].copy()
            new_row['cantidad'] = pred
            new_row['fecha'] += pd.DateOffset(months=1)
            new_row['month'] = new_row['fecha'].dt.month
            new_row['year'] = new_row['fecha'].dt.year
            
            current_data = pd.concat([current_data, new_row])
            
            if feature_columns:
                for lag in range(1, 4):
                    current_data[f'cantidad_lag_{lag}'] = current_data['cantidad'].shift(lag)
        
        return predictions

    def make_final_predictions(self, all_monthly_data, df_cov, df_correlaciones_sig):
        results = []
        no_sku_process_list = []
        count = 0
        for sku in all_monthly_data['articulo'].unique():
            print(f"Processing SKU: {sku}")
            try:
                sku_data = all_monthly_data[all_monthly_data['articulo'] == sku].copy()
                last_date = sku_data['fecha'].max()
                last_date = (last_date + pd.offsets.MonthBegin(1)).normalize()
                lt = int(sku_data['lt'].iloc[-1])
                last_year = sku_data['year'].max()

                # if len(sku_data) < 7:
                #     continue

                data, feature_columns = self.prepare_features_for_ml(
                    all_monthly_data, df_cov, df_correlaciones_sig, sku
                )

                best_config, best_model, score = self.evaluate_models(
                    data,
                    feature_columns,
                    lookback_periods=[6, 12, None],
                    val_year=last_year - 1
                )

                # if best_config is None:
                #     continue

                if best_config is None or best_model is None:
                    results.append({
                        'sku': sku,
                        'lt': lt,
                        'date': last_date,
                        'model': np.nan,
                        'real': 0,
                        'catusita': np.nan,
                        'lookback_period': np.nan,
                        'features_used': 'none',
                        'caa': np.nan,
                        'caa_lt': np.nan,
                        'corr_sd': np.nan,
                        'loss': np.nan
                    })
                    print(f"SKU {sku} no pudo ser evaluado. Datos insuficientes o problema en los datos.")
                    no_sku_process_list.append({'sku':sku})
                    # pd.DataFrame(no_sku_process_list,columns=['sku']).to_csv('data/cleaned/no_sku_process_list.csv')
                    count = count + 1
                    continue

                best_model_name, lookback = best_config

                # Calculate test score
                test_data = data[data['year'] == last_year]
                if feature_columns:
                    test_X = test_data[feature_columns]
                    if best_model_name in ['xgboost', 'linear']:
                        test_pred = best_model.predict(test_X)
                else:
                    if best_model_name in ['mean', 'median']:
                        test_pred = [best_model] * len(test_data)
                    else:  # ES
                        forecast_func, alpha = best_model
                        test_pred = []
                        current_data = data[data['year'] < last_year]['cantidad'].values
                        for _ in range(len(test_data)):
                            pred = forecast_func(current_data, alpha)
                            test_pred.append(pred)
                            current_data = np.append(current_data, pred)

                
                test_score = self.weighted_mape(test_data['cantidad'], test_pred)

                # Generate future predictions
                future_predictions = self.predict_future_months(
                    data,
                    feature_columns,
                    best_model,
                    best_model_name,
                    lookback,
                    2 * lt
                )

                # Calculate pred_std from the historical data used for prediction
                if lookback:
                    historical_data = data['cantidad'].iloc[-lookback:]
                else:
                    historical_data = data['cantidad']
                pred_std = np.std(historical_data)

                period1 = sum(future_predictions[:lt])
                period2 = sum(future_predictions[lt:2*lt])

                # period2 += 0.4 * pred_std

                last_six_months_mean = sku_data.tail(6)['cantidad'].mean()

                results.append({
                    'sku': sku,
                    'lt': lt,
                    'date': last_date,
                    'model': best_model_name,
                    'real': 0,
                    'catusita': last_six_months_mean,
                    'lookback_period': lookback if lookback else 'all',
                    'features_used': ','.join(feature_columns) if feature_columns else 'none',
                    'caa': period1,
                    'caa_lt': period2,
                    'corr_sd': pred_std,
                    'loss': test_score
                })

            except Exception as e:
                print(f"Error processing SKU {sku}: {str(e)}")
                continue
        print(f"El numero de SKU sin procesar: {count}")
        return pd.DataFrame(results)

    def process_predictions(self):
        from utils.process_data.config import DATA_PATHS
        
        data_path = DATA_PATHS['process'] / 'catusita_consolidated.csv'
        cov_path = DATA_PATHS['process'] / 'df_covariables.csv'
        
        all_monthly_data = self.create_monthly_sales_data(
            self.load_and_preprocess_data(data_path)
        )
        df_cov = self.load_covariates_data(cov_path)
        df_correlaciones_sig = pd.read_csv(DATA_PATHS['process'] / 'df_correlaciones_sig.csv')
        
        results_df = self.make_final_predictions(
            all_monthly_data, 
            df_cov, 
            df_correlaciones_sig
        )

        # results_df = results_df[results_df['date'] == results_df['date'].max()]

        if not results_df.empty:
            return results_df.sort_values('loss')
        return None


In [13]:
from utils.process_data.config import DATA_PATHS
# from utils.predictions.predictor import Predictor
predictor = Predictor()
# predictions_df = predictor.process_predictions()

data_path = DATA_PATHS['process'] / 'catusita_consolidated.csv'
cov_path = DATA_PATHS['process'] / 'df_covariables.csv'

all_monthly_data = predictor.create_monthly_sales_data(
    predictor.load_and_preprocess_data(data_path)
)
df_cov = predictor.load_covariates_data(cov_path)
df_correlaciones_sig = pd.read_csv(DATA_PATHS['process'] / 'df_correlaciones_sig.csv')

# results_df = predictor.make_final_predictions(
#     all_monthly_data, 
#     df_cov, 
#     df_correlaciones_sig
# )

In [15]:
sku = 'ctx-170z'

In [6]:
all_monthly_data[all_monthly_data['articulo']==sku].sort_values('fecha', ascending=False)['fecha'].unique()

<DatetimeArray>
['2024-12-01 00:00:00', '2024-11-01 00:00:00', '2024-10-01 00:00:00',
 '2024-09-01 00:00:00', '2024-08-01 00:00:00', '2024-07-01 00:00:00',
 '2024-06-01 00:00:00', '2024-05-01 00:00:00', '2024-04-01 00:00:00',
 '2024-02-01 00:00:00', '2024-01-01 00:00:00', '2023-12-01 00:00:00',
 '2023-10-01 00:00:00', '2023-09-01 00:00:00', '2023-08-01 00:00:00',
 '2023-07-01 00:00:00', '2023-06-01 00:00:00']
Length: 17, dtype: datetime64[ns]

In [61]:
df_cov.head()

,year,month,Hibridos y Electricos,Livianos,Menores,Pesados,Remolques y SemiR,Neumáticos,Lubricantes,Partes de Motor,...,Sistema de frenos,Baterías,Sistema de suspensión,Accesorios,Ruedas y sus partes,Productos de caucho,Sistema de dirección,Sistema de enfriamiento,Ejes y diferencial,Sistema de escape
0,2020,1,51.0,14420.0,25197.0,1381.0,410.0,4.222718e+07,2.762023e+07,2.656240e+07,...,4106407.664,3827787.802,2862318.494,2486253.950,1526067.344,2104315.510,1828623.462,1200065.896,1454949.952,514298.718
1,2020,2,53.0,12753.0,22494.0,1137.0,334.0,2.819364e+07,1.924647e+07,2.266911e+07,...,3332410.839,2906918.541,3075611.878,2302485.475,1898314.536,1730286.225,1530739.708,978654.869,1065694.594,692901.929
2,2020,3,42.0,7084.0,12592.0,580.0,183.0,2.003285e+07,2.594233e+07,1.490268e+07,...,2327674.710,1932872.809,2024623.605,1919918.731,1037609.573,1260022.243,693044.502,381527.233,859822.498,264183.002
3,2020,4,42.0,7084.0,12592.0,580.0,183.0,3.007109e+07,2.570013e+07,1.250405e+07,...,2458361.309,1391027.834,1896856.959,1117306.921,883819.533,1015749.498,622813.906,518195.523,634857.940,235871.411
4,2020,5,42.0,348.0,185.0,27.0,7.0,2.850129e+07,1.241780e+07,1.236305e+07,...,1914440.701,1587255.421,1132746.440,1670601.475,1511083.002,1219568.677,622659.173,549842.197,426656.089,134154.035


In [64]:
df_correlaciones_sig[df_correlaciones_sig['sku']==sku].head()

,lag,tipo,corr,sku
110920,0,Hibridos y Electricos,0.000,ctx-170z
110921,0,Livianos,0.000,ctx-170z
110922,0,Menores,0.000,ctx-170z
110923,0,Pesados,0.000,ctx-170z
110924,0,Remolques y SemiR,-0.428,ctx-170z


In [20]:
results = []
try:
    print(f" \n estamos viendo resultados para {sku} \n")
    sku_data = all_monthly_data[all_monthly_data['articulo'] == sku].copy()
    last_date = sku_data['fecha'].max()
    last_date = (last_date + pd.offsets.MonthBegin(1)).normalize()
    lt = int(sku_data['lt'].iloc[-1])
    last_year = sku_data['year'].max()

    lookback_periods=[6, 12, None]
    val_year=last_year - 1

    data, feature_columns = predictor.prepare_features_for_ml(
        all_monthly_data, df_cov, df_correlaciones_sig, sku
    )

    # best_config, best_model, score = predictor.evaluate_models(
    #     data,
    #     feature_columns,
    #     lookback_periods=[6, 12, None],
    #     val_year=last_year - 1
    # )

    ## empieza evaluate_models()

    best_score = float('inf')
    best_config = None
    best_model = None
    
    # if len(data) < 4:
    #     return None, None, float('inf')
        
    # val_data = data[data['year'] == val_year] if val_year else data
    val_data = data
    val_size = 6
    print(data['year'].min())
    for lookback in lookback_periods:
        train_data = (val_data.iloc[-(lookback+val_size):-val_size] 
                    if lookback else val_data.iloc[:-val_size])
        test_data = val_data.iloc[-val_size:]
        print(f"\n ========== Estamos en el lookback {lookback} con un val_size {val_size} de tal forma que ==========\n")
        print(f"el train_data tiene fechas unicas {train_data['fecha'].unique()} y el test_data {test_data['fecha'].unique()} ==========\n")

        
        # if len(train_data) < 2:
        #     continue
            
        y_train = train_data['cantidad']
        y_test = test_data['cantidad']
        
        models = {
            'mean': y_train.mean(),
            'median': y_train.median(),
            'es': (predictor.ES_forecast, predictor.ES_opt_alpha(y_train))
        }
        print(f"por lo tanto el mean y median del train_data['cantidad'] son {models['mean']} y {models['median']}\n")
        
        if len(feature_columns) > 0:
            models.update({
                'xgboost': xgb.XGBRegressor(random_state=42),
                'linear': LinearRegression()
            })
            X_train = train_data[feature_columns]
            X_test = test_data[feature_columns]
        
        for model_name, model in models.items():
            try:
                if model_name in ['xgboost', 'linear']:
                    model.fit(X_train, y_train)
                    y_pred = model.predict(X_test)
                elif model_name == 'es':
                    forecast_func, alpha = model
                    y_pred = []
                    current_data = y_train.values
                    for _ in range(len(y_test)):
                        pred = forecast_func(current_data, alpha)
                        y_pred.append(pred)
                        current_data = np.append(current_data, pred)
                else:  # mean or median
                    y_pred = [model] * len(y_test)
                # print(f"y_pred antes de trucar: {y_pred}")
                y_pred = np.maximum(y_pred, 0)
                # print(f"y_pred despues de trucar: {y_pred}")
                score = predictor.weighted_mape(y_test, y_pred)
                print(f"el modelo {model_name} tiene score igual a {score}")
                
                if score < best_score:
                    best_score = score
                    best_config = (model_name, lookback)
                    best_model = model
                    
            except Exception as e:
                continue
except Exception as e:
    print(f"Error processing SKU {sku}: {str(e)}")

 
 estamos viendo resultados para ctx-170z 

2021

 ========== Estamos en el lookback 6 con un val_size 6 de tal forma que ==========

el train_data tiene fechas unicas <DatetimeArray>
['2023-12-01 00:00:00', '2024-01-01 00:00:00', '2024-02-01 00:00:00',
 '2024-04-01 00:00:00', '2024-05-01 00:00:00', '2024-06-01 00:00:00']
Length: 6, dtype: datetime64[ns] y el test_data <DatetimeArray>
['2024-07-01 00:00:00', '2024-08-01 00:00:00', '2024-09-01 00:00:00',
 '2024-10-01 00:00:00', '2024-11-01 00:00:00', '2024-12-01 00:00:00']
Length: 6, dtype: datetime64[ns] ==========

por lo tanto el mean y median del train_data['cantidad'] son 273.3333333333333 y 175.5

el modelo mean tiene score igual a 758.0652519747368
el modelo median tiene score igual a 514.05955374652
el modelo es tiene score igual a 789.1212042598348
el modelo xgboost tiene score igual a 966.8189697265625
el modelo linear tiene score igual a 2482.5176153106736

 ========== Estamos en el lookback 12 con un val_size 6 de tal forma

In [19]:
data['fecha'].unique()

<DatetimeArray>
['2021-10-01 00:00:00', '2021-11-01 00:00:00', '2022-02-01 00:00:00',
 '2022-04-01 00:00:00', '2022-05-01 00:00:00', '2022-07-01 00:00:00',
 '2022-08-01 00:00:00', '2022-09-01 00:00:00', '2022-10-01 00:00:00',
 '2022-11-01 00:00:00', '2022-12-01 00:00:00', '2023-01-01 00:00:00',
 '2023-02-01 00:00:00', '2023-03-01 00:00:00', '2023-04-01 00:00:00',
 '2023-05-01 00:00:00', '2023-06-01 00:00:00', '2023-07-01 00:00:00',
 '2023-08-01 00:00:00', '2023-09-01 00:00:00', '2023-10-01 00:00:00',
 '2023-12-01 00:00:00', '2024-01-01 00:00:00', '2024-02-01 00:00:00',
 '2024-04-01 00:00:00', '2024-05-01 00:00:00', '2024-06-01 00:00:00',
 '2024-07-01 00:00:00', '2024-08-01 00:00:00', '2024-09-01 00:00:00',
 '2024-10-01 00:00:00', '2024-11-01 00:00:00', '2024-12-01 00:00:00']
Length: 33, dtype: datetime64[ns]

In [19]:
results = []
try:
    sku_data = all_monthly_data[all_monthly_data['articulo'] == sku].copy()
    last_date = sku_data['fecha'].max()
    last_date = (last_date + pd.offsets.MonthBegin(1)).normalize()
    lt = int(sku_data['lt'].iloc[-1])
    last_year = sku_data['year'].max()

    # if len(sku_data) < 7:
    #     continue

    data, feature_columns = predictor.prepare_features_for_ml(
        all_monthly_data, df_cov, df_correlaciones_sig, sku
    )

    best_config, best_model, score = predictor.evaluate_models(
        data,
        feature_columns,
        lookback_periods=[6, 12, None],
        val_year=last_year - 1
    )

    # if best_config is None:
    #     continue

    if best_config is None or best_model is None:
        results.append({
            'sku': sku,
            'lt': lt,
            'date': last_date,
            'model': np.nan,
            'real': 0,
            'catusita': np.nan,
            'lookback_period': np.nan,
            'features_used': 'none',
            'caa': np.nan,
            'caa_lt': np.nan,
            'corr_sd': np.nan,
            'loss': np.nan
        })
        print(f"SKU {sku} no pudo ser evaluado. Datos insuficientes o problema en los datos.")
        
    best_model_name, lookback = best_config

    # # Calculate test score
    # test_data = data[data['year'] == last_year]
    # if feature_columns:
    #     test_X = test_data[feature_columns]
    #     if best_model_name in ['xgboost', 'linear']:
    #         test_pred = best_model.predict(test_X)
    # else:
    #     if best_model_name in ['mean', 'median']:
    #         test_pred = [best_model] * len(test_data)
    #     else:  # ES
    #         forecast_func, alpha = best_model
    #         test_pred = []
    #         current_data = data[data['year'] < last_year]['cantidad'].values
    #         for _ in range(len(test_data)):
    #             pred = forecast_func(current_data, alpha)
    #             test_pred.append(pred)
    #             current_data = np.append(current_data, pred)

    # test_score = predictor.weighted_mape(test_data['cantidad'], test_pred)

    # # Generate future predictions
    # future_predictions = predictor.predict_future_months(
    #     data,
    #     feature_columns,
    #     best_model,
    #     best_model_name,
    #     lookback,
    #     2 * lt
    # )

    # def predict_future_months(self, data, feature_columns, best_model, 
    #                         best_model_name, lookback, num_months):
    num_months = 2 * lt
    predictions = []
    current_data = data.copy()
    current_data = current_data.sort_values(by="fecha")
    # print(current_data[-6:])

    # if best_model_name == 'linear':
    #     coeficientes = best_model.coef_
    #     intercepto = best_model.intercept_
        
    #     # Construimos la ecuación del modelo
    #     ecuacion = f"cantidad_predicha = {intercepto:.10f}"
    #     for coef, feature in zip(coeficientes, feature_columns):
    #         ecuacion += f"\n + ({coef:.10f} * {feature})"
        
    #     print(f"\nLa forma funcional del modelo lineal es:\n{ecuacion}\n")
        
    for i in range(num_months):
        # Extraer valores de las variables predictoras
        feature_values = current_data[feature_columns].iloc[-1:].to_dict(orient='records')[0]

        print(f"\n\nValores, coeficientes y contribuciones para el mes {i+1} de prediccion:")

        if best_model_name == 'linear':
            coeficientes = best_model.coef_
            intercepto = best_model.intercept_

            suma_parcial = 0  # Para acumular la suma de coef * valor
            for feature, coef in zip(feature_columns, coeficientes):
                valor = feature_values[feature]
                contribucion = coef * valor
                suma_parcial += contribucion
                print(f"{feature}: {valor:.4f}  |  Coeficiente: {coef:.4f}  |  Contribución: {contribucion:.4f}")

            # Imprimir suma total de las contribuciones sin incluir la intersección
            print(f"Suma parcial (sin intercepto): {suma_parcial:.4f}")

            # Sumar el intercepto
            pred = suma_parcial + intercepto
            print(f"Intercepto: {intercepto:.4f}")
            print(f"Predicción final para mes {i+1}: {pred:.4f}")

        else:
            if best_model_name in ['xgboost']:
                pred = best_model.predict(current_data[feature_columns].iloc[-1:])[0]
            elif best_model_name in ['mean', 'median']:
                last_6_months = current_data.iloc[-6:]  # Tomar solo los últimos 6 meses
                pred = last_6_months['cantidad'].mean() if best_model_name == 'mean' else last_6_months['cantidad'].median()
            else:  # ES
                forecast_func, alpha = best_model
                lookback_data = (current_data.iloc[-lookback:] if lookback else current_data)
                pred = forecast_func(lookback_data['cantidad'].dropna(), alpha)

        pred = np.maximum(pred, 0)
        predictions.append(pred)

        new_row = current_data.iloc[-1:].copy()
        new_row['cantidad'] = pred
        new_row['fecha'] += pd.DateOffset(months=1)
        new_row['month'] = new_row['fecha'].dt.month
        new_row['year'] = new_row['fecha'].dt.year

        current_data = pd.concat([current_data, new_row])

        if feature_columns:
            for lag in range(1, 4):
                current_data[f'cantidad_lag_{lag}'] = current_data['cantidad'].shift(lag)


except Exception as e:
    print(f"Error processing SKU {sku}: {str(e)}")




Valores, coeficientes y contribuciones para el mes 1 de prediccion:
Hibridos y Electricos: 552.0000  |  Coeficiente: 0.0000  |  Contribución: 0.0000
Livianos: 12212.0000  |  Coeficiente: -0.0000  |  Contribución: -0.0000
Menores: 21692.0000  |  Coeficiente: 0.0000  |  Contribución: 0.0002
Hibridos y Electricos_lag_1: 454.0000  |  Coeficiente: -0.0000  |  Contribución: -0.0000
Livianos_lag_1: 25506.0000  |  Coeficiente: 0.0000  |  Contribución: 0.0000
Menores_lag_1: 56760.0000  |  Coeficiente: 0.0000  |  Contribución: 0.0004
Pesados_lag_1: 3436.0000  |  Coeficiente: -0.0000  |  Contribución: -0.0000
Hibridos y Electricos_lag_2: 497.0000  |  Coeficiente: 0.0000  |  Contribución: 0.0000
Livianos_lag_2: 26240.0000  |  Coeficiente: 0.0000  |  Contribución: 0.0006
Menores_lag_2: 63398.0000  |  Coeficiente: 0.0000  |  Contribución: 0.0021
Pesados_lag_2: 3000.0000  |  Coeficiente: 0.0000  |  Contribución: 0.0000
Hibridos y Electricos_lag_3: 597.0000  |  Coeficiente: -0.0000  |  Contribución:

In [ ]:
import requests
import pandas as pd 

# URL de la API
url = "http://api.catusita.com:8083/api/sales/forDate"
# http://api.catusita.com:8083/api/sales/forDate?Date1=20250101&Date2=20250115

# Parámetros de la consulta
params = {
    "Date1": "20250101",
    "Date2": "20250223"
}

# Headers opcionales (descomentar si son necesarios)
headers = {
    # "Authorization": "Bearer <token>",  # Si se requiere autenticación
    "Accept": "application/json"  # Para indicar que esperamos respuesta en JSON
}

try:
    # Realizar la solicitud GET con headers
    response = requests.get(url, params=params, headers=headers)

    # Verificar el estado de la respuesta
    response.raise_for_status()  # Lanza un error si el código de estado no es 200

    # Intentar obtener la clave "data" del JSON
    json_response = response.json()
    
    if "data" in json_response and isinstance(json_response["data"], list):
        data = pd.DataFrame(json_response["data"])
        print("Datos obtenidos correctamente:")
        print(data.head())  # Muestra las primeras filas
    else:
        print("Advertencia: La respuesta no contiene datos válidos.")
        data = pd.DataFrame()

except requests.exceptions.RequestException as e:
    print(f"Error en la solicitud: {e}")
    data = pd.DataFrame()


In [4]:
data.head()

# dateDocument:fecha, codeArticle: articulo, nameArticle: nombre, nameSupply: fuente_suministro, quantity: cantidad, amountSOL: venta_pen, amountUSD: venta_usd

,dateDocument,document,codeArticle,nameArticle,codeSupply,nameSupply,quantity,amountSOL,amountUSD,cost
0,2025-02-21T00:00:00,F001-01-0060719,RXFC113641B,LIQUIDO DE FRENO DOT3 1LT 32 oz 946ml LATA,C310,WAGNER LOCKED - USA,24,562.41,152.54,479.089992
1,2025-01-13T00:00:00,F001-01-0059747,RXFC113640BEP,LIQUIDO DE FRENO DOT3 12 onzas PLASTICO,C310,WAGNER LOCKED - USA,24,214.64,56.95,125.700000
2,2025-01-13T00:00:00,F001-01-0059747,RXFC113644BEP,LIQUIDO DE FRENO DOT4 1LT 32 onzas PLASTICO,c310,WAGNER LOCKED - USA,24,651.58,172.88,402.600000
3,2025-01-13T00:00:00,F001-01-0059747,RXFC113641BEP,LIQUIDO DE FRENO DOT3 1LT 32 oz 946ml PLASTICO,C310,WAGNER LOCKED - USA,24,459.93,122.03,280.630008
4,2025-02-18T00:00:00,F001-01-0060623,M25600100,ACEITE SAE25W60 SL MAXPOWER,A003,VALVOLINE,12,735.33,198.31,489.570000


In [ ]:
import requests
import pandas as pd

# http://api.catusita.com:8083/api/stock/forDate?DateStock=20250210
url = "http://api.catusita.com:8083/api/stock/forDate"

# Parámetros de la consulta
params = {
    "DateStock": "20241231"
}


# Encabezados opcionales (puedes agregar más si la API lo requiere)
headers = {
    "Accept": "application/json"  # Asegura que la API devuelva JSON
}

try:
    # Hacer la solicitud GET a la API
    response = requests.get(url, params=params, headers=headers, timeout=10)

    # Verificar si la respuesta fue exitosa
    response.raise_for_status()  # Lanza un error si la respuesta no es 200

    # Intentar convertir la respuesta en JSON
    try:
        data = response.json()
        # print(data)  # Imprime los datos obtenidos
    except ValueError:
        print("⚠️ La respuesta no es un JSON válido.")

except requests.exceptions.RequestException as e:
    print(f"❌ Error al conectar con la API: {e}")


# Extraer la lista de datos desde el JSON
data_list = data.get("data", [])  # Asegura que no falle si la clave "data" no existe

# Convertir a DataFrame
df2 = pd.DataFrame(data_list)

In [ ]:
df2.head()
# ['codigo':'codeArticle','date': es la fecha que se coloca para extraer la informacion (En este caso 
# ya no se considerara porque se usara la fecha del ultimo dia del mes correspondiente previo a la fecha de prediccion),
# 'um': , 'stock':'umArticle']

,codeArticle,umArticle,stock
0,SFC-7912-30B,PZA,73
1,DT195/55R15,PZA,20
2,MWMWC440,JGO,4
3,55351-2E501,PZA,19
4,O-1325,PZA,356


In [40]:
import requests
import pandas as pd

# http://api.catusita.com:8083/api/article/data
# Nuevo URL de la API
url = "http://api.catusita.com:8083/api/article/data"

# Parámetros de la consulta (ajústalos según sea necesario)
params = {
    "DateStock": "20252402"  # Asegúrate de que este parámetro sea válido para la nueva API
}

# Encabezados opcionales (puedes agregar más si la API lo requiere)
headers = {
    "Accept": "application/json"  # Asegura que la API devuelva JSON
}

try:
    # Hacer la solicitud GET a la API
    response = requests.get(url, params=params, headers=headers, timeout=10)

    # Verificar si la respuesta fue exitosa
    response.raise_for_status()  # Lanza un error si la respuesta no es 200

    # Intentar convertir la respuesta en JSON
    try:
        data = response.json()
        print(data)  # Imprime los datos obtenidos para ver su estructura
    except ValueError:
        print("⚠️ La respuesta no es un JSON válido.")

except requests.exceptions.RequestException as e:
    print(f"❌ Error al conectar con la API: {e}")

# Extraer la lista de datos desde el JSON (ajusta la clave según la estructura de la respuesta)
data_list = data.get("data", [])  # Asegura que no falle si la clave "data" no existe

# Convertir a DataFrame
df = pd.DataFrame(data_list)

{'data': [{'codeSupply': 'A003', 'nameSupply': 'VALVOLINE', 'codeArticle': 'B10300025', 'nameArticle': 'ACEITE SAE10W30 SN SEMI SINTETICO', 'unitArticle': 'QT', 'lastCurrencyBuy': 'USD', 'lastPriceBuy': 3.73, 'factPrice': 3.429566, 'lastDateBuy': '2024-11-12T00:00:00', 'lastQuantityBuy': 432}, {'codeSupply': 'A003', 'nameSupply': 'VALVOLINE', 'codeArticle': 'B10300100', 'nameArticle': 'ACEITE SAE10W30 SN SEMI SINTETICO', 'unitArticle': 'GLN', 'lastCurrencyBuy': 'USD', 'lastPriceBuy': 12.99, 'factPrice': 3.351316, 'lastDateBuy': '2024-11-12T00:00:00', 'lastQuantityBuy': 1050}, {'codeSupply': 'A003', 'nameSupply': 'VALVOLINE', 'codeArticle': 'B20500100', 'nameArticle': 'ACEITE SAE20W50 SN SEMI SINTETICO', 'unitArticle': 'GLN', 'lastCurrencyBuy': 'USD', 'lastPriceBuy': 12.99, 'factPrice': 0, 'lastDateBuy': '2024-11-12T00:00:00', 'lastQuantityBuy': 300}, {'codeSupply': 'A003', 'nameSupply': 'VALVOLINE', 'codeArticle': 'DI1540CI100', 'nameArticle': 'ACEITE SAE15W40 CI4 PLUS/SL', 'unitArticl

In [14]:
len(df['codeArticle'].drop_duplicates())

17225

In [67]:
# df['codeArticle'].astype(str).sort_values().to_csv('articulos.csv')
df[df['codeArticle']=='SFC-79410-30S']
# ['Código':'codeArticle','Mnd':'lastCurrencyBuy','Fob':'factPrice','Ult. Fecha':'lastDateBuy','Ult. Compra':'lastQuantityBuy']

,codeSupply,nameSupply,codeArticle,nameArticle,unitArticle,lastCurrencyBuy,lastPriceBuy,factPrice,lastDateBuy,lastQuantityBuy
2999,A017,SAKURA FILTER,SFC-79410-30S,FILTRO SEP DE AGUA(12)VG1092080052,PZA,USD,14.94,0.0,2024-09-17T00:00:00,36.0


In [24]:
df_tc = pd.read_excel("C:/Users/YOGA/Desktop/repositories/caa/catusita/catusita_predictions/data/raw/catusita/saldo de todo 04.11.2024.2.xls", skiprows=2)

In [68]:
df['codeArticle'] = df['codeArticle'].str.lower()

In [70]:
df[df['codeArticle']=='sfc-79410-30s']

,codeSupply,nameSupply,codeArticle,nameArticle,unitArticle,lastCurrencyBuy,lastPriceBuy,factPrice,lastDateBuy,lastQuantityBuy
2999,A017,SAKURA FILTER,sfc-79410-30s,FILTRO SEP DE AGUA(12)VG1092080052,PZA,USD,14.94,0.0,2024-09-17T00:00:00,36.0
